# Tc Extraction Visualisation

Visualisations of critical temperature (Tc) data extracted from superconductor papers via **text extraction** and **VLM figure reading**.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import re

# Project style
SRC_DIR = str(Path().resolve().parents[2] / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from llm_synthesis.utils.style_utils import set_style, get_palette
set_style("manuscript")

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.grid": False,
})
sns.set_style("white")  # override seaborn grid

PAL = get_palette()
# Assign semantic roles from the palette
C_BLUE    = PAL[2]   # #448FF2
C_ORANGE  = PAL[4]   # #F9CB9C
C_PURPLE  = PAL[12]  # #7B5AEF
C_PINK    = PAL[0]   # #E0A2D3
C_LBLUE   = PAL[3]   # #A7C8F2
C_LPURPLE = PAL[13]  # #B27EDD
C_GREY    = PAL[8]   # #D3D3D3
C_DGREY   = PAL[10]  # #808080
C_BLACK   = PAL[11]  # #000000
C_LAVENDER = PAL[1]  # #DFD6FB

print("Style loaded, grid disabled")

## Data Loading & Cleanup

The raw CSV has two issues we fix before plotting:
1. **Junk material names** — entries like `x = 0.08`, `0.5 T`, `100 µA`, `pristine (x=0.107)` etc. that are conditions/labels, not real chemical formulas
2. **Duplicate materials** — the same compound extracted from text vs VLM gets different names (e.g. `MgB2` from VLM vs the LaTeX-normalised `MgB2` from text with condition `x=0.0`). We normalise formulas and merge rows within each paper so text + VLM values are combined.

In [ ]:
DATA_DIR = Path("../../data/results_Tc_v2_20_02")
df_raw = pd.read_csv(DATA_DIR / "tc_master_snippet.csv")
print(f"Raw CSV: {len(df_raw)} rows, {df_raw['paper_id'].nunique()} papers")
df_raw.head(3)

### Step 1 — Remove junk entries

Filter out rows where `material_normalized` is not a real chemical formula but rather a condition label (doping level, magnetic field, current, pressure, irradiation dose, etc.).

### Step 2 — Normalise formulas & merge duplicates

Within each paper, the same compound can appear with slightly different names:
- `LaFeAsO0.89F0.11` (VLM row) vs `LaFeAs(O1-xFx)` + condition `x=0.11` (text row)  
- `MgB2` (VLM row) vs `MgB2` (text row with different condition string)

We normalise formulas by:
1. Stripping LaTeX markup, subscript/superscript unicode, and whitespace
2. Expanding generic `(O1-xFx)` patterns using the `x=...` in the condition column
3. Grouping by `(paper_id, normalised_formula)` and merging Tc values (first non-null wins for each column)

In [ ]:
# --- Formula normalisation ---

# Unicode subscript/superscript -> ASCII
_UNICODE_MAP = str.maketrans(
    "₀₁₂₃₄₅₆₇₈₉⁰¹²³⁴⁵⁶⁷⁸⁹ₓ",
    "01234567890123456789x",
)


def normalise_formula(name: str, condition: str = "") -> str:
    """
    Normalise a material formula to a canonical ASCII string for matching.
    
    Steps:
      1. Strip LaTeX commands  ($, \\mathrm{}, _{}, ^{})
      2. Convert unicode sub/superscripts to ASCII
      3. Remove decoration like colour hints "(red circles)" etc.
      4. If formula contains generic 'x' variable AND condition contains x=<value>,
         substitute to produce the concrete formula.
      5. Strip whitespace
    """
    s = str(name)

    # Strip LaTeX
    s = re.sub(r"\$", "", s)
    s = re.sub(r"\\mathrm\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\[a-zA-Z]+\{([^}]*)\}", r"\1", s)  # generic \cmd{...}
    s = re.sub(r"[_^]\{([^}]*)\}", r"\1", s)           # _{...} ^{...}
    s = re.sub(r"[_^](.)", r"\1", s)                     # _X ^X single char

    # Unicode sub/superscripts
    s = s.translate(_UNICODE_MAP)

    # Remove colour/legend hints: "(red circles)", "(blue)", "(olive/brown)"
    s = re.sub(r"\s*\([^)]*(?:circle|square|blue|red|green|yellow|brown|olive|teal|dash)[^)]*\)", "", s, flags=re.IGNORECASE)

    # Remove trailing whitespace
    s = s.strip()

    # Try to substitute x variable from condition
    cond = str(condition)
    x_match = re.search(r"x\s*=\s*([\d.]+)", cond)
    if x_match and "x" in s.lower():
        x_val = float(x_match.group(1))
        one_minus_x = round(1 - x_val, 4)
        # Replace patterns like (O1-xFx) or (O_{1-x}F_x) etc.
        # Generic: replace "1-x" with the numeric value, then "x" with x_val
        s_expanded = s.replace("1-x", str(one_minus_x))
        # Careful: only replace standalone 'x' (not inside element names like "Ox")
        # We replace x that appears after a letter or ( and before a letter or ) or digit
        s_expanded = re.sub(r"(?<=[A-Za-z(])x(?=[A-Za-z)0-9]|$)", str(x_val), s_expanded)
        s = s_expanded

    # Remove ± and delta indicators
    s = s.replace("±", "").replace("delta", "").replace("δ", "")

    # Collapse whitespace
    s = re.sub(r"\s+", "", s)

    return s


# Apply normalisation
df_clean["formula_norm"] = df_clean.apply(
    lambda r: normalise_formula(r["material_normalized"], r.get("condition", "")), axis=1
)

# Show normalisation mapping for debugging
norm_map = df_clean[["material_normalized", "condition", "formula_norm"]].drop_duplicates()
print(f"Unique raw names: {df_clean['material_normalized'].nunique()}")
print(f"Unique normalised: {df_clean['formula_norm'].nunique()}")
print("\nNormalisation examples (where raw != normalised):")
changed = norm_map[norm_map["material_normalized"] != norm_map["formula_norm"]]
for _, row in changed.head(20).iterrows():
    print(f"  {row['material_normalized']:45s}  cond={str(row['condition'])[:25]:25s}  →  {row['formula_norm']}")

In [ ]:
# --- Merge duplicate rows within each paper ---

TC_COLS = ["tc_text", "tc_text_onset", "tc_text_zero",
           "tc_vlm_orig", "tc_vlm_orig_onset", "tc_vlm_orig_zero",
           "tc_vlm_snip", "tc_vlm_snip_onset", "tc_vlm_snip_zero",
           "tc_best"]


def merge_group(grp: pd.DataFrame) -> pd.Series:
    """Merge multiple rows for the same (paper, formula) into one."""
    # Take the first row as base
    base = grp.iloc[0].copy()

    # For Tc columns: take the first non-null value across all rows
    for col in TC_COLS:
        if col in grp.columns:
            vals = grp[col].dropna()
            base[col] = vals.iloc[0] if len(vals) > 0 else np.nan

    # For boolean flags: OR across rows (if any row has it, the merged row has it)
    for col in ["has_text_tc", "has_vlm_tc_orig", "has_vlm_tc_snip"]:
        if col in grp.columns:
            base[col] = grp[col].any()

    # For is_superconductor: True if any row says True
    if "is_superconductor" in grp.columns:
        base["is_superconductor"] = grp["is_superconductor"].any()

    # Recompute tc_best: prefer text, then vlm_snip, then vlm_orig
    tc_candidates = [
        ("text", base.get("tc_text")),
        ("vlm_snippet", base.get("tc_vlm_snip")),
        ("vlm_original", base.get("tc_vlm_orig")),
        ("text_onset", base.get("tc_text_onset")),
    ]
    for src, val in tc_candidates:
        if pd.notna(val):
            base["tc_best"] = val
            base["tc_best_source"] = src
            break
    else:
        base["tc_best"] = np.nan
        base["tc_best_source"] = "none"

    # Merge source descriptions
    sources = grp["tc_text_source"].dropna().unique()
    base["tc_text_source"] = "; ".join(sources) if len(sources) > 0 else np.nan

    # Keep the most informative material name (longest real formula)
    base["material_normalized"] = grp["material_normalized"].iloc[
        grp["material_normalized"].str.len().argmax()
    ]

    # Merge conditions
    conds = grp["condition"].dropna().unique()
    base["condition"] = "; ".join(str(c) for c in conds) if len(conds) > 0 else np.nan

    return base


df_merged = (
    df_clean
    .groupby(["paper_id", "formula_norm"], sort=False)
    .apply(merge_group)
    .reset_index(drop=True)
)

n_before = len(df_clean)
n_after = len(df_merged)
n_merged_away = n_before - n_after
print(f"Before merge: {n_before} rows")
print(f"After merge:  {n_after} rows  ({n_merged_away} rows merged)")

# Show which entries got merged
if n_merged_away > 0:
    print("\nMerged groups:")
    for (pid, fnorm), grp in df_clean.groupby(["paper_id", "formula_norm"], sort=False):
        if len(grp) > 1:
            names = grp["material_normalized"].unique()
            if len(names) > 1 or len(grp) > 1:
                print(f"  {pid} | {fnorm}: {len(grp)} rows  ←  {list(names)}")

### Step 3 — Save cleaned CSV & prepare for plotting

In [ ]:
# Save cleaned CSV
out_path = DATA_DIR / "tc_master_snippet_cleaned.csv"
df_merged.to_csv(out_path, index=False)
print(f"Saved cleaned CSV to {out_path}")

# ---- From here on, `df` is the cleaned dataframe ----
df = df_merged.copy()


def extract_year_from_arxiv_id(paper_id: str) -> int:
    """Extract publication year from arXiv ID (old format YYMMNNN or new format YYMM.NNNNN)."""
    first = paper_id.split("_")[0]
    match = re.match(r"(\d{2})", first)
    if match:
        yy = int(match.group(1))
        return 2000 + yy if yy < 90 else 1900 + yy
    return None


df["year"] = df["paper_id"].apply(extract_year_from_arxiv_id)


# Classify material families
def classify_family(mat: str) -> str:
    mat_lower = mat.lower()
    if "mgb" in mat_lower or "alb" in mat_lower or re.search(r"\w+b2", mat_lower):
        return "Borides (MgB$_2$-type)"
    if "feas" in mat_lower or "feaso" in mat_lower:
        return "Iron pnictides"
    if "fese" in mat_lower or "fete" in mat_lower or re.search(r"fe.*te.*se", mat_lower) or re.search(r"fe.*se", mat_lower):
        return "Iron chalcogenides"
    if "cu" in mat_lower and ("ba" in mat_lower or "sr" in mat_lower or "la" in mat_lower or "ca" in mat_lower):
        return "Cuprates"
    if "bis" in mat_lower or "bise" in mat_lower:
        return "BiS$_2$-based"
    if "re" in mat_lower and "mo" in mat_lower:
        return "Re-Mo alloys"
    if "hote" in mat_lower or ("pd" in mat_lower and "te" in mat_lower):
        return "Tellurides"
    if "ta" in mat_lower and "pd" in mat_lower and "s" in mat_lower:
        return "Chalcogenides (Ta$_2$PdS$_5$)"
    if "w-c" in mat_lower:
        return "W-C FIBID"
    if "wp" in mat_lower or "crp" in mat_lower or "cop" in mat_lower:
        return "Transition metal phosphides"
    return "Other"


df["family"] = df["material_normalized"].apply(classify_family)

# Convert boolean columns
for col in ["has_text_tc", "has_vlm_tc_orig", "has_vlm_tc_snip"]:
    df[col] = df[col].astype(bool)

# Filter to superconductors
sc = df[df["is_superconductor"] == True].copy()

print(f"\n{'='*50}")
print(f"CLEANED dataset ready for plotting")
print(f"  Total records:   {len(df)}")
print(f"  Superconductors: {len(sc)}")
print(f"  Unique papers:   {df['paper_id'].nunique()}")
print(f"  Year range:      {df['year'].min()} – {df['year'].max()}")
print(f"  Material families: {df['family'].nunique()}")
print(f"{'='*50}")
df.head()

---
## 1 · Tc vs Publication Year

In [ ]:
plot_df = sc.dropna(subset=["tc_best"]).copy()

fig, ax = plt.subplots(figsize=(8, 4.5))

# Color by source of tc_best — using palette colors
source_colors = {"text": C_BLUE, "vlm_snippet": C_PURPLE, "vlm_original": C_PINK,
                 "text_onset": C_LBLUE, "none": C_GREY}
source_labels = {"text": "Text", "vlm_snippet": "VLM (snippet)", "vlm_original": "VLM (original)",
                 "text_onset": "Text (onset)", "none": "None"}

for src, grp in plot_df.groupby("tc_best_source"):
    color = source_colors.get(src, C_DGREY)
    label = source_labels.get(src, src)
    ax.scatter(grp["year"], grp["tc_best"], c=color, label=label,
              edgecolors="k", linewidths=0.3, s=50, alpha=0.85, zorder=3)

# Annotate a few notable materials
for _, row in plot_df.iterrows():
    if row["tc_best"] > 90 or (row["material_normalized"] == "MgB2" and row["condition"] == "ambient"):
        ax.annotate(row["material_normalized"], (row["year"], row["tc_best"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=7.5, fontstyle="italic")

ax.set_xlim(plot_df["year"].min() - 1, plot_df["year"].max() + 2)
ax.set_xlabel("Publication year")
ax.set_ylabel("$T_c$ (K)")
#ax.set_title("Extracted $T_c$ vs Publication Year")
ax.legend(title="Best $T_c$ source", fontsize=8, title_fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Only text-extracted Tc values
plot_df = sc.dropna(subset=["tc_text"]).copy()

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(plot_df["year"], plot_df["tc_text"], c=C_BLUE,
           edgecolors="k", linewidths=0.3, s=50, alpha=0.85, zorder=3)

# Annotate a few notable materials
for _, row in plot_df.iterrows():
    if row["tc_text"] > 90 or (row["material_normalized"] == "MgB2" and row["condition"] == "ambient"):
        ax.annotate(row["material_normalized"], (row["year"], row["tc_text"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=7.5, fontstyle="italic")

ax.set_xlim(plot_df["year"].min() - 1, plot_df["year"].max() + 2)
ax.set_ylim(-2, plot_df["tc_text"].max() * 1.15)
ax.set_xlabel("Publication year", fontsize=18)
ax.set_ylabel("$T_c$ (K)", fontsize=18)

# Remove top and right spines (no box)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

---
## 2 · Text-extracted Tc vs VLM-extracted Tc

In [ ]:
# Use tc_vlm_snip as primary VLM value, fall back to tc_vlm_orig
sc["tc_vlm_any"] = sc["tc_vlm_snip"].fillna(sc["tc_vlm_orig"])

both = sc.dropna(subset=["tc_text", "tc_vlm_any"]).copy()
max_tc = max(both["tc_text"].max(), both["tc_vlm_any"].max()) * 1.1

fig, ax = plt.subplots(figsize=(5, 5))

sns.regplot(x="tc_text", y="tc_vlm_any", data=both, ci=95,
            scatter_kws={"s": 45, "edgecolors": "k", "linewidths": 0.3, "alpha": 0.8, "zorder": 3,
                         "color": C_BLUE},
            line_kws={"lw": 1.5, "color": C_PURPLE}, ax=ax)

# Identity line
ax.plot([0, max_tc], [0, max_tc], "k--", lw=1, alpha=0.5, label="$y = x$")

# Stats
r2 = both[["tc_text", "tc_vlm_any"]].corr().iloc[0, 1] ** 2
mae = (both["tc_text"] - both["tc_vlm_any"]).abs().mean()
ax.text(0.05, 0.92, f"$R^2$ = {r2:.3f}\nMAE = {mae:.1f} K\n$n$ = {len(both)}",
        transform=ax.transAxes, fontsize=9, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Show text-only and VLM-only as marginal markers
text_only = sc[sc["tc_text"].notna() & sc["tc_vlm_any"].isna()]
vlm_only = sc[sc["tc_text"].isna() & sc["tc_vlm_any"].notna()]
ax.scatter(text_only["tc_text"], [-1.5] * len(text_only), c=C_PURPLE, marker="v", s=25,
           alpha=0.7, label=f"Text only (n={len(text_only)})", zorder=2)
ax.scatter([-1.5] * len(vlm_only), vlm_only["tc_vlm_any"], c=C_PINK, marker="<", s=25,
           alpha=0.7, label=f"VLM only (n={len(vlm_only)})", zorder=2)

ax.set_xlim(-4, max_tc)
ax.set_ylim(-4, max_tc)
ax.set_xlabel("$T_c$ from text (K)")
ax.set_ylabel("$T_c$ from VLM (K)")
ax.set_title("Text vs VLM $T_c$ Extraction")
ax.legend(fontsize=7.5, loc="lower right")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Pearson R²: {r2:.3f}")
print(f"MAE: {mae:.2f} K")
print(f"Matched: {len(both)}, Text-only: {len(text_only)}, VLM-only: {len(vlm_only)}")

---
## 3 · Data Source Coverage

In [ ]:
# Count materials by extraction source combination
cats = {
    "Text only":       sc[sc["has_text_tc"] & ~sc["has_vlm_tc_orig"] & ~sc["has_vlm_tc_snip"]].shape[0],
    "VLM orig only":   sc[~sc["has_text_tc"] & sc["has_vlm_tc_orig"] & ~sc["has_vlm_tc_snip"]].shape[0],
    "VLM snip only":   sc[~sc["has_text_tc"] & ~sc["has_vlm_tc_orig"] & sc["has_vlm_tc_snip"]].shape[0],
    #"Text + VLM orig": sc[sc["has_text_tc"] & sc["has_vlm_tc_orig"] & ~sc["has_vlm_tc_snip"]].shape[0],
    #"Text + VLM snip": sc[sc["has_text_tc"] & ~sc["has_vlm_tc_orig"] & sc["has_vlm_tc_snip"]].shape[0],
    "VLM orig + snip": sc[~sc["has_text_tc"] & sc["has_vlm_tc_orig"] & sc["has_vlm_tc_snip"]].shape[0],
    "All three":       sc[sc["has_text_tc"] & sc["has_vlm_tc_orig"] & sc["has_vlm_tc_snip"]].shape[0],
    "None":            sc[~sc["has_text_tc"] & ~sc["has_vlm_tc_orig"] & ~sc["has_vlm_tc_snip"]].shape[0],
}

# Filter to non-zero
cats = {k: v for k, v in cats.items() if v > 0}

colors_cov = [C_BLUE, C_PINK, C_PURPLE, C_LPURPLE, C_LBLUE, C_LAVENDER, C_ORANGE, C_GREY]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
bars = ax1.barh(list(cats.keys()), list(cats.values()), color=colors_cov[:len(cats)], edgecolor="k", linewidth=0.3)
for bar, val in zip(bars, cats.values()):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             str(val), va="center", fontsize=9)
ax1.set_xlabel("Number of superconductor entries")
ax1.set_title("Extraction Source Combinations")

# Summary pie: Text vs VLM vs Both vs None
has_any_text = sc["has_text_tc"].sum()
has_any_vlm = (sc["has_vlm_tc_orig"] | sc["has_vlm_tc_snip"]).sum()
has_both = (sc["has_text_tc"] & (sc["has_vlm_tc_orig"] | sc["has_vlm_tc_snip"])).sum()
has_none = (~sc["has_text_tc"] & ~sc["has_vlm_tc_orig"] & ~sc["has_vlm_tc_snip"]).sum()

pie_data = {
    f"Text only\n({has_any_text - has_both})": has_any_text - has_both,
    f"VLM only\n({has_any_vlm - has_both})": has_any_vlm - has_both,
    f"Both\n({has_both})": has_both,
    f"Neither\n({has_none})": has_none,
}
pie_colors = [C_BLUE, C_PURPLE, C_PINK, C_GREY]
ax2.pie(pie_data.values(), labels=pie_data.keys(), colors=pie_colors,
        autopct=lambda p: f"{p:.0f}%" if p > 3 else "",
        startangle=90, wedgeprops={"edgecolor": "k", "linewidth": 0.3})
ax2.set_title("Text vs VLM Coverage (superconductors)")

plt.tight_layout()
plt.show()

---
## 4 · VLM Original vs VLM Snippet

In [ ]:
vlm_both = sc.dropna(subset=["tc_vlm_orig", "tc_vlm_snip"]).copy()
max_vlm = max(vlm_both["tc_vlm_orig"].max(), vlm_both["tc_vlm_snip"].max()) * 1.1

fig, ax = plt.subplots(figsize=(5, 5))

sns.regplot(x="tc_vlm_orig", y="tc_vlm_snip", data=vlm_both, ci=95,
            scatter_kws={"s": 45, "edgecolors": "k", "linewidths": 0.3, "alpha": 0.8, "zorder": 3,
                         "color": C_PURPLE},
            line_kws={"lw": 1.5, "color": C_PINK}, ax=ax)

ax.plot([0, max_vlm], [0, max_vlm], "k--", lw=1, alpha=0.5, label="$y = x$")

r2_vlm = vlm_both[["tc_vlm_orig", "tc_vlm_snip"]].corr().iloc[0, 1] ** 2
mae_vlm = (vlm_both["tc_vlm_orig"] - vlm_both["tc_vlm_snip"]).abs().mean()
ax.text(0.05, 0.92, f"$R^2$ = {r2_vlm:.3f}\nMAE = {mae_vlm:.1f} K\n$n$ = {len(vlm_both)}",
        transform=ax.transAxes, fontsize=9, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax.set_xlim(-1, max_vlm)
ax.set_ylim(-1, max_vlm)
ax.set_xlabel("$T_c$ from VLM (full figure) (K)")
ax.set_ylabel("$T_c$ from VLM (cropped snippet) (K)")
ax.set_title("VLM: Full Figure vs Cropped Snippet")
ax.legend(fontsize=8)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Pearson R²: {r2_vlm:.3f}")
print(f"MAE: {mae_vlm:.2f} K")
print(f"n = {len(vlm_both)}")

---
## 5 · Bland-Altman: Text vs VLM Agreement

In [ ]:
ba = sc.dropna(subset=["tc_text", "tc_vlm_any"]).copy()
ba["mean_tc"] = (ba["tc_text"] + ba["tc_vlm_any"]) / 2
ba["diff_tc"] = ba["tc_text"] - ba["tc_vlm_any"]

mean_diff = ba["diff_tc"].mean()
std_diff = ba["diff_tc"].std()

fig, ax = plt.subplots(figsize=(6, 4))

ax.scatter(ba["mean_tc"], ba["diff_tc"], s=45, c=C_BLUE,
           edgecolors="k", linewidths=0.3, alpha=0.8, zorder=3)

ax.axhline(mean_diff, color=C_PURPLE, ls="-", lw=1.2, label=f"Mean diff = {mean_diff:.1f} K")
ax.axhline(mean_diff + 1.96 * std_diff, color=C_PINK, ls="--", lw=0.8, alpha=0.8,
           label=f"+1.96 SD = {mean_diff + 1.96 * std_diff:.1f} K")
ax.axhline(mean_diff - 1.96 * std_diff, color=C_PINK, ls="--", lw=0.8, alpha=0.8,
           label=f"$-$1.96 SD = {mean_diff - 1.96 * std_diff:.1f} K")
ax.axhline(0, color=C_DGREY, ls=":", lw=0.8, alpha=0.5)

ax.set_xlabel("Mean of Text & VLM $T_c$ (K)")
ax.set_ylabel("Difference (Text $-$ VLM) (K)")
ax.set_title("Bland-Altman: Text vs VLM Agreement")
ax.legend(fontsize=7.5)
plt.tight_layout()
plt.show()

print(f"Mean difference: {mean_diff:.2f} K")
print(f"Std of difference: {std_diff:.2f} K")
print(f"95% limits of agreement: [{mean_diff - 1.96*std_diff:.1f}, {mean_diff + 1.96*std_diff:.1f}] K")

---
## 6 · Tc Distribution by Material Family

In [ ]:
fam_df = sc.dropna(subset=["tc_best"]).copy()
# Keep families with at least 2 entries
family_counts = fam_df["family"].value_counts()
keep_families = family_counts[family_counts >= 2].index
fam_df = fam_df[fam_df["family"].isin(keep_families)]

# Sort by median Tc
order = fam_df.groupby("family")["tc_best"].median().sort_values(ascending=False).index

# Build a palette from our style colors for the families
fam_palette = {fam: PAL[i % len(PAL)] for i, fam in enumerate(order)}

fig, ax = plt.subplots(figsize=(8, 4.5))

sns.boxplot(data=fam_df, x="tc_best", y="family", order=order,
            palette=fam_palette, width=0.6, linewidth=0.8, fliersize=3, ax=ax)
sns.stripplot(data=fam_df, x="tc_best", y="family", order=order,
              color=C_BLACK, size=3.5, alpha=0.5, jitter=0.15, ax=ax)

ax.set_xlabel("$T_c$ (K)")
ax.set_ylabel("")
ax.set_title("$T_c$ Distribution by Material Family")
plt.tight_layout()
plt.show()

---
## 7 · Per-Paper Extraction Yield

In [ ]:
# For each paper, count materials with text Tc, VLM Tc, both, or neither
papers = []
for pid, grp in df.groupby("paper_id"):
    year = grp["year"].iloc[0]
    n_total = len(grp)
    has_text = grp["has_text_tc"].sum()
    has_vlm = (grp["has_vlm_tc_orig"] | grp["has_vlm_tc_snip"]).sum()
    has_both = (grp["has_text_tc"] & (grp["has_vlm_tc_orig"] | grp["has_vlm_tc_snip"])).sum()
    text_only = has_text - has_both
    vlm_only = has_vlm - has_both
    neither = n_total - text_only - vlm_only - has_both
    papers.append({
        "paper_id": pid, "year": year,
        "Text only": text_only, "VLM only": vlm_only,
        "Both": has_both, "Neither": neither,
    })

paper_df = pd.DataFrame(papers).sort_values("year")
paper_df["label"] = paper_df["paper_id"].str.split("_").str[0] + " (" + paper_df["year"].astype(str) + ")"

fig, ax = plt.subplots(figsize=(10, 5))

stack_cols = ["Both", "Text only", "VLM only", "Neither"]
stack_colors = [C_PINK, C_BLUE, C_PURPLE, C_GREY]

bottoms = np.zeros(len(paper_df))
for col, color in zip(stack_cols, stack_colors):
    ax.barh(paper_df["label"], paper_df[col], left=bottoms, label=col,
            color=color, edgecolor="k", linewidth=0.3)
    bottoms += paper_df[col].values

ax.set_xlabel("Number of materials")
ax.set_title("Extraction Yield per Paper")
ax.legend(loc="lower right", fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 8 · Synthesis Methods (from `results_Tc_synthesis`)

Load the larger dataset from `results_Tc_synthesis/tc_master.csv` which includes the `synthesis_method` field extracted from the JSON outputs. Count materials per synthesis method.

In [ ]:
# Load the synthesis dataset (separate, larger dataset with synthesis_method column)
SYNTH_DIR = Path("../../data/results_Tc_synthesis")
df_synth = pd.read_csv(SYNTH_DIR / "tc_master.csv")
print(f"Synthesis dataset: {len(df_synth)} rows, {df_synth['paper_id'].nunique()} papers")
# Drop rows without a synthesis method
df_synth = df_synth.dropna(subset=["synthesis_method"])
df_synth = df_synth[df_synth["synthesis_method"].str.strip() != ""]
print(f"With synthesis method: {len(df_synth)} rows")
# Normalise method names (lowercase, strip whitespace)
df_synth["synthesis_method_clean"] = df_synth["synthesis_method"].str.strip().str.lower()
# Count per method, sorted
method_counts = df_synth["synthesis_method_clean"].value_counts()
print(f"\nUnique synthesis methods: {len(method_counts)}")
print(method_counts.to_string())
# --- Horizontal bar chart ---
fig, ax = plt.subplots(figsize=(7, 7))
order = method_counts.index[::-1]  # ascending so largest at top
counts = method_counts.loc[order]
bars = ax.barh(
    range(len(order)), counts.values,
    color=C_PURPLE, edgecolor="k", linewidth=0.3
)
# Annotate counts on bars
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=13)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([m.title() for m in order], fontsize=14)
ax.set_xlabel("Number of materials", fontsize=15)
ax.set_ylabel("")  # no y-label needed, but if you add one: fontsize=15
ax.set_title("Synthesis Methods Across Superconductor Papers", fontsize=16)
ax.tick_params(axis="x", labelsize=14)
ax.set_xlim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()
